# Lecture 4: Model Training, Gradients, and Backpropagation


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

np.random.seed(42)
print("Libraries loaded.")


## 1. Gradient Descent (Ch. 6.1)

Parameter update rule:

$$\\theta \\leftarrow \\theta - \\alpha \\frac{\\partial L}{\\partial \\theta}$$

`α` = learning rate

A step is taken in the direction of steepest descent on the loss surface.


In [ ]:
# Visualizing GD on a 2D loss surface
def loss_surface(w1, w2):
    return (w1 - 1)**2 + 4*(w2 - 2)**2  # elliptic paraboloid

w1_range = np.linspace(-2, 4, 100)
w2_range = np.linspace(-1, 5, 100)
W1, W2 = np.meshgrid(w1_range, w2_range)
L = loss_surface(W1, W2)

def gradient(w1, w2):
    dw1 = 2*(w1 - 1)
    dw2 = 8*(w2 - 2)
    return np.array([dw1, dw2])

# GD path: different learning rates
lrs = [0.05, 0.2, 0.45]
colors = ['limegreen', 'steelblue', 'crimson']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, lr, color in zip(axes, lrs, colors):
    ax.contourf(W1, W2, L, levels=25, cmap='YlOrRd', alpha=0.8)
    ax.contour(W1, W2, L, levels=25, colors='white', linewidths=0.5, alpha=0.5)
    
    w = np.array([-1.5, 4.5])
    path = [w.copy()]
    for _ in range(40):
        g = gradient(*w)
        w = w - lr * g
        path.append(w.copy())
    path = np.array(path)
    
    ax.plot(path[:, 0], path[:, 1], '-o', color=color, markersize=3, lw=1.5, label=f"lr={lr}")
    ax.plot(1, 2, 'w*', markersize=15, zorder=5)
    ax.set_title(f"Gradient Descent (lr={lr})")
    ax.set_xlabel("w1"); ax.set_ylabel("w2")
    ax.legend(loc='upper right')

plt.suptitle("Effect of Different Learning Rates", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb04_gd.png", dpi=100, bbox_inches='tight')
plt.show()


## 2. Stochastic and Mini-Batch Gradient Descent (Ch. 6.2)

| Method | Gradient Computed Over | Advantage | Disadvantage |
|---|---|---|---|
| Batch GD | Full dataset | Stable direction | Slow for large data |
| SGD | 1 sample | Fast updates | Noisy path |
| Mini-Batch | B samples | Balanced | Requires batch size tuning |


In [ ]:
np.random.seed(7)
N = 500
X = np.random.randn(N, 1)
y_true = 2*X.flatten() + 1 + np.random.randn(N)*0.5

def compute_loss_grad(X, y, w, b):
    y_hat = X.flatten() * w + b
    loss = np.mean((y_hat - y)**2)
    dw = np.mean(2*(y_hat - y)*X.flatten())
    db = np.mean(2*(y_hat - y))
    return loss, dw, db

# Compare 3 methods
methods = [
    ("Batch GD (N=500)", 500),
    ("Mini-Batch (B=32)", 32),
    ("SGD (B=1)", 1),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for method_name, batch_size in methods:
    w, b = 0.0, 0.0
    lr = 0.05
    losses_ep = []
    for epoch in range(40):
        idxs = np.random.permutation(N)
        ep_loss = 0
        n_batches = 0
        for start in range(0, N, batch_size):
            batch_idx = idxs[start:start+batch_size]
            Xb, yb = X[batch_idx], y_true[batch_idx]
            loss, dw, db = compute_loss_grad(Xb, yb, w, b)
            w -= lr * dw
            b -= lr * db
            ep_loss += loss
            n_batches += 1
        losses_ep.append(ep_loss / n_batches)
    axes[0].plot(losses_ep, label=method_name, lw=1.5)

axes[0].set_title("Per-Epoch Loss Comparison")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_yscale('log')

# Mini-batch noise visualization
w2, b2 = 0.0, 0.0
step_losses = []
for _ in range(200):
    idx = np.random.choice(N, 32, replace=False)
    loss, dw, db = compute_loss_grad(X[idx], y_true[idx], w2, b2)
    w2 -= 0.05 * dw; b2 -= 0.05 * db
    step_losses.append(loss)

axes[1].plot(step_losses, 'steelblue', lw=1, alpha=0.7, label="Per-step loss")
smoothed = np.convolve(step_losses, np.ones(10)/10, mode='valid')
axes[1].plot(smoothed, 'crimson', lw=2, label="Smoothed (10-step)")
axes[1].set_title("Mini-Batch SGD: Noisy Loss Path")
axes[1].set_xlabel("Step"); axes[1].set_ylabel("MSE")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_nb04_sgd.png", dpi=100, bbox_inches='tight')
plt.show()


## 3. Momentum and Adam (Ch. 6.3, 6.4)

**Momentum:** Accumulates past gradients as 'velocity'
$$v \\leftarrow \\beta v + (1-\\beta)g, \\quad \\theta \\leftarrow \\theta - \\alpha v$$

**Adam:** Combines first- and second-moment estimates
$$m \\leftarrow \\beta_1 m + (1-\\beta_1)g$$
$$v \\leftarrow \\beta_2 v + (1-\\beta_2)g^2$$
$$\\theta \\leftarrow \\theta - \\alpha \\frac{\\hat{m}}{\\sqrt{\\hat{v}}+\\epsilon}$$


In [ ]:
# Optimizer comparison on the Rosenbrock function
def rosenbrock(x, y, a=1, b=10):
    return (a - x)**2 + b*(y - x**2)**2

def rosenbrock_grad(x, y, a=1, b=10):
    dx = -2*(a - x) - 4*b*x*(y - x**2)
    dy = 2*b*(y - x**2)
    return np.array([dx, dy])

def run_optimizer(name, steps=500):
    w = np.array([-1.5, 1.5], dtype=float)
    lr = 0.002
    path = [w.copy()]
    
    if name == "GD":
        for _ in range(steps):
            g = rosenbrock_grad(*w)
            w -= lr * g
            path.append(w.copy())
    
    elif name == "Momentum":
        v = np.zeros(2)
        beta = 0.9
        for _ in range(steps):
            g = rosenbrock_grad(*w)
            v = beta*v + (1-beta)*g
            w -= lr * v
            path.append(w.copy())
    
    elif name == "Adam":
        m, v2 = np.zeros(2), np.zeros(2)
        b1, b2, eps = 0.9, 0.999, 1e-8
        for t in range(1, steps+1):
            g = rosenbrock_grad(*w)
            m = b1*m + (1-b1)*g
            v2 = b2*v2 + (1-b2)*g**2
            m_hat = m / (1-b1**t)
            v_hat = v2 / (1-b2**t)
            w -= lr * m_hat / (np.sqrt(v_hat)+eps)
            path.append(w.copy())
    
    return np.array(path)

x_r = np.linspace(-2, 2, 200)
y_r = np.linspace(-0.5, 3.5, 200)
Xr, Yr = np.meshgrid(x_r, y_r)
Zr = np.log(rosenbrock(Xr, Yr) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
optimizer_styles = [("GD", 'steelblue'), ("Momentum", 'darkorange'), ("Adam", 'crimson')]

for ax, (opt_name, color) in zip(axes, optimizer_styles):
    path = run_optimizer(opt_name)
    ax.contourf(Xr, Yr, Zr, levels=30, cmap='viridis', alpha=0.8)
    ax.plot(path[:, 0], path[:, 1], '-', color=color, lw=1.5, alpha=0.8)
    ax.plot(path[0, 0], path[0, 1], 'wo', markersize=8, label="Start")
    ax.plot(1, 1, 'r*', markersize=12, label="Minimum (1,1)")
    final_loss = rosenbrock(*path[-1])
    ax.set_title(f"{opt_name}\nFinal loss: {final_loss:.4f}")
    ax.set_xlabel("w1"); ax.set_ylabel("w2")
    ax.legend(fontsize=7)

plt.suptitle("Optimizer Comparison on Rosenbrock Function", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb04_optimizers.png", dpi=100, bbox_inches='tight')
plt.show()


## 4. Backpropagation (Ch. 7)

Backprop efficiently computes the gradient of every parameter via the **chain rule**.

**Forward pass:** Compute and cache intermediate values  
**Backward pass:** Propagate gradients back via chain rule

$$\\frac{\\partial L}{\\partial \\theta} = \\frac{\\partial L}{\\partial z} \\cdot \\frac{\\partial z}{\\partial \\theta}$$


In [ ]:
# Simple 2-layer network - manual backprop
# Architecture: x -> [W1,b1] -> ReLU -> [W2,b2] -> y_hat -> MSE loss

class SimpleNet:
    def __init__(self, n_in=2, n_hidden=4, n_out=1):
        np.random.seed(0)
        self.W1 = np.random.randn(n_hidden, n_in) * 0.5
        self.b1 = np.zeros(n_hidden)
        self.W2 = np.random.randn(n_out, n_hidden) * 0.5
        self.b2 = np.zeros(n_out)
        self.cache = {}
    
    def forward(self, X):
        z1 = X @ self.W1.T + self.b1
        h1 = np.maximum(0, z1)
        z2 = h1 @ self.W2.T + self.b2
        self.cache = {'X': X, 'z1': z1, 'h1': h1, 'z2': z2}
        return z2
    
    def backward(self, y):
        X = self.cache['X']
        z1 = self.cache['z1']
        h1 = self.cache['h1']
        z2 = self.cache['z2']
        N = X.shape[0]
        
        # Output layer gradient
        dL_dz2 = 2*(z2 - y.reshape(-1,1)) / N
        dL_dW2 = dL_dz2.T @ h1
        dL_db2 = dL_dz2.sum(axis=0)
        
        # Hidden layer gradient
        dL_dh1 = dL_dz2 @ self.W2
        dL_dz1 = dL_dh1 * (z1 > 0)  # ReLU derivative
        dL_dW1 = dL_dz1.T @ X
        dL_db1 = dL_dz1.sum(axis=0)
        
        return {'W1': dL_dW1, 'b1': dL_db1, 'W2': dL_dW2, 'b2': dL_db2}
    
    def update(self, grads, lr=0.01):
        self.W1 -= lr * grads['W1']
        self.b1 -= lr * grads['b1']
        self.W2 -= lr * grads['W2']
        self.b2 -= lr * grads['b2']

# Training
np.random.seed(1)
X_data = np.random.randn(100, 2)
y_data = X_data[:, 0]**2 + X_data[:, 1] * 0.5 + 0.3

net = SimpleNet(n_in=2, n_hidden=8, n_out=1)
losses = []
for epoch in range(500):
    y_hat = net.forward(X_data)
    loss = np.mean((y_hat.flatten() - y_data)**2)
    losses.append(loss)
    grads = net.backward(y_data)
    net.update(grads, lr=0.05)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(losses, 'steelblue', lw=2)
axes[0].set_title("Training with Manual Backprop")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE Loss")
axes[0].set_yscale('log'); axes[0].grid(alpha=0.3)

y_pred_final = net.forward(X_data).flatten()
axes[1].scatter(y_data, y_pred_final, alpha=0.5, color='steelblue', s=20)
axes[1].plot([y_data.min(), y_data.max()], [y_data.min(), y_data.max()], 'r--', lw=2)
axes[1].set_title("True vs Predicted")
axes[1].set_xlabel("True y"); axes[1].set_ylabel("Predicted y")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_nb04_backprop.png", dpi=100, bbox_inches='tight')
plt.show()
print(f"Final loss: {losses[-1]:.6f}")


## 5. Parameter Initialization (Ch. 7.5)

Poor initialization → **gradient explosion / vanishing**.

**He initialization** (for ReLU):
$$W \\sim \\mathcal{N}\\left(0, \\sqrt{\\frac{2}{n_{\\text{in}}}}\\right)$$

**Xavier/Glorot initialization** (for sigmoid/tanh):
$$W \\sim \\mathcal{U}\\left(-\\sqrt{\\frac{6}{n_{\\text{in}}+n_{\\text{out}}}}, \\sqrt{\\frac{6}{n_{\\text{in}}+n_{\\text{out}}}}\\right)$$


In [ ]:
def simulate_signal_propagation(n_layers=8, n_neurons=256, init_type='standard'):
    x = np.random.randn(1000, n_neurons)
    std_per_layer = [x.std()]
    
    for layer in range(n_layers):
        if init_type == 'standard':
            W = np.random.randn(n_neurons, n_neurons) * 1.0
        elif init_type == 'small':
            W = np.random.randn(n_neurons, n_neurons) * 0.01
        elif init_type == 'he':
            W = np.random.randn(n_neurons, n_neurons) * np.sqrt(2.0/n_neurons)
        elif init_type == 'xavier':
            limit = np.sqrt(6.0/(n_neurons+n_neurons))
            W = np.random.uniform(-limit, limit, (n_neurons, n_neurons))
        
        x = np.maximum(0, x @ W)  # ReLU
        std_per_layer.append(x.std())
    
    return std_per_layer

fig, ax = plt.subplots(figsize=(10, 5))

for init, color, label in [
    ('standard', 'crimson', 'Standard (std=1)'),
    ('small', 'gray', 'Too small (std=0.01)'),
    ('he', 'steelblue', 'He Initialization'),
    ('xavier', 'darkorange', 'Xavier Initialization'),
]:
    stds = simulate_signal_propagation(init_type=init)
    ax.plot(stds, '-o', color=color, lw=2, markersize=5, label=label)

ax.set_title("Activation Std Dev Across Layers — Effect of Initialization", fontsize=13)
ax.set_xlabel("Layer"); ax.set_ylabel("Std Dev (activation)")
ax.set_yscale('log'); ax.legend(); ax.grid(alpha=0.3)
ax.axhline(1, color='green', linestyle=':', lw=1.5, label="Ideal: std=1")
plt.tight_layout()
plt.savefig("fig_nb04_init.png", dpi=100, bbox_inches='tight')
plt.show()


## 6. Summary

| Concept | Description |
|---|---|
| **Gradient Descent** | Update parameters in the gradient direction to reduce the loss |
| **SGD** | Compute gradient on mini-batch; fast but noisy |
| **Momentum** | Accumulate past gradients to reduce oscillation |
| **Adam** | Adaptive learning rate; generally the best default choice |
| **Backprop** | Compute gradients for all parameters via the chain rule |
| **He initialization** | Ideal for ReLU; preserves activation variance |

**Next Notebook →** Performance Measurement & Regularization
